# 数据分析

这本 notebook 用来分析 `minimal_adl_ethene_butadiene/` 这条主线产生的关键结果文件，既适合回看本次第一轮结果，也适合后续第二轮、第三轮继续复用。

**默认分析的文件类型**

- 配置文件：`configs/base.yaml`
- delta 数据集：`delta_dataset.npz` 与 `delta_dataset_metadata.json`
- 训练结果：`train_main_status.json`、`train_aux_status.json`、`training_summary.json`
- 不确定性结果：`uncertainty_latest.json`
- 轮次选点结果：`round_001_selection_summary.json`

**使用提醒**

- notebook 会优先按仓库当前约定路径自动找文件。
- 如果某个文件暂时不存在，相关单元会给出中文提示并跳过，不会让整本 notebook 直接报错。
- 如果你以后分析第二轮或换体系，最后一节会说明优先改哪些路径或文件名。


## Notebook 结构

这份模板按“先检查文件，再看总览，再看细节”的顺序组织：

1. 路径与文件检查
2. 本次实验概览
3. delta 数据集基本统计
4. 训练结果分析
5. 不确定性分析
6. 第 1 轮选点分析
7. 面向汇报的自动结论
8. 复用说明


## 0. 准备分析环境

这一格负责导入常用库、自动定位仓库根目录，并定义后面各节都会复用的读取函数。这样 notebook 从上到下运行时，不依赖隐藏状态，也更容易迁移到别的轮次继续用。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

# 统一图表风格，并兼容无界面环境。
if plt is not None:
    plt.switch_backend("Agg")
    plt.rcParams["figure.figsize"] = (8, 4.5)
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
else:
    print("提示：当前环境缺少 matplotlib，图形会跳过，但统计表仍可继续生成。")

# 让表格输出更容易阅读。
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

# 自动寻找仓库根目录，避免 notebook 只能在固定目录运行。
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "minimal_adl_ethene_butadiene" / "configs" / "base.yaml").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / "minimal_adl_ethene_butadiene"


# 尽量把路径显示成相对仓库根目录的形式，方便阅读。
def as_relative(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(path.resolve())


# 有文件就读取，没有就返回 None，并给出清晰提示。
def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        print(f"[缺失] {as_relative(path)}")
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[读取失败] {as_relative(path)}: {type(exc).__name__}: {exc}")
        return None


# 读取 YAML 配置文件，用于提取默认参数。
def load_yaml_if_exists(path: Path) -> dict | None:
    if not path.exists():
        print(f"[缺失] {as_relative(path)}")
        return None
    try:
        return yaml.safe_load(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[读取失败] {as_relative(path)}: {type(exc).__name__}: {exc}")
        return None


# 把 npz 文件读取成普通字典，后面分析时更方便。
def load_npz_if_exists(path: Path) -> dict[str, np.ndarray] | None:
    if not path.exists():
        print(f"[缺失] {as_relative(path)}")
        return None
    try:
        with np.load(path, allow_pickle=True) as payload:
            return {key: payload[key] for key in payload.files}
    except Exception as exc:
        print(f"[读取失败] {as_relative(path)}: {type(exc).__name__}: {exc}")
        return None


# 统计 manifest 里的样本数。
def count_manifest_samples(payload: dict | None) -> int | None:
    if not isinstance(payload, dict):
        return None
    samples = payload.get("samples")
    if isinstance(samples, list):
        return len(samples)
    return None


# 把布尔值转成更适合汇报的中文。
def bool_to_text(value) -> str:
    if value is True:
        return "是"
    if value is False:
        return "否"
    return "未知"


# 按顺序取第一个非空值，方便做结果文件与默认配置的回退。
def first_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None


PATHS = {
    "config": PROJECT_DIR / "configs" / "base.yaml",
    "delta_metadata": PROJECT_DIR / "data" / "processed" / "delta_dataset_metadata.json",
    "train_main_status": PROJECT_DIR / "models" / "train_main_status.json",
    "train_aux_status": PROJECT_DIR / "models" / "train_aux_status.json",
    "training_summary": PROJECT_DIR / "models" / "training_summary.json",
    "uncertainty": PROJECT_DIR / "results" / "uncertainty_latest.json",
    "round_selection": PROJECT_DIR / "results" / "round_001_selection_summary.json",
}

OPTIONAL_PATHS = {
    "geometry_pool_manifest": PROJECT_DIR / "data" / "raw" / "geometry_pool_manifest.json",
    "initial_selection_manifest": PROJECT_DIR / "data" / "raw" / "initial_selection_manifest.json",
    "delta_npz": PROJECT_DIR / "data" / "processed" / "delta_dataset.npz",
}

TOP_N = 10

print(f"仓库根目录: {REPO_ROOT}")
print(f"分析对象目录: {PROJECT_DIR}")
print("模板已加载，可以继续做文件检查。")


## 1. 路径与文件检查

这一格先检查 notebook 默认会读取的关键文件是否存在。这样你一眼就能知道当前工作区是“完整可分析”，还是“只有部分结果，某些章节会跳过”。


In [ ]:
# 把必查文件和补充文件合并成一张检查表，方便统一查看。
file_rows = []

for label, path in PATHS.items():
    file_rows.append(
        {
            "类别": "必查",
            "文件": label,
            "路径": as_relative(path),
            "是否存在": "是" if path.exists() else "否",
        }
    )

for label, path in OPTIONAL_PATHS.items():
    file_rows.append(
        {
            "类别": "补充",
            "文件": label,
            "路径": as_relative(path),
            "是否存在": "是" if path.exists() else "否",
        }
    )

file_check_df = pd.DataFrame(file_rows)

# 给出更友好的提示，而不是缺文件就中断。
missing_required = file_check_df[(file_check_df["类别"] == "必查") & (file_check_df["是否存在"] == "否")]
if not missing_required.empty:
    print("提示：当前有必查文件缺失，后续相关章节会自动跳过。")
    print("缺失文件：" + "、".join(missing_required["文件"].tolist()))
else:
    print("必查文件已全部找到，可以做完整分析。")

file_check_df


## 2. 本次实验概览

这一格会尽量把“这次到底跑到了哪一步”总结成一张总览表。优先用真实结果文件；如果某些结果文件缺失，就回退到配置文件中的默认值，并在“来源”列里标出来。


In [ ]:
# 先把后面会反复用到的文件一次性读入，避免每格重复读文件。
config = load_yaml_if_exists(PATHS["config"])
delta_metadata = load_json_if_exists(PATHS["delta_metadata"])
train_main_status = load_json_if_exists(PATHS["train_main_status"])
train_aux_status = load_json_if_exists(PATHS["train_aux_status"])
training_summary = load_json_if_exists(PATHS["training_summary"])
uncertainty_payload = load_json_if_exists(PATHS["uncertainty"])
selection_summary = load_json_if_exists(PATHS["round_selection"])

geometry_pool_manifest = load_json_if_exists(OPTIONAL_PATHS["geometry_pool_manifest"])
initial_selection_manifest = load_json_if_exists(OPTIONAL_PATHS["initial_selection_manifest"])
delta_dataset = load_npz_if_exists(OPTIONAL_PATHS["delta_npz"])

# 记录关键数量，并说明这些数是来自结果文件还是来自配置默认值。
pool_size_from_manifest = count_manifest_samples(geometry_pool_manifest)
pool_size = first_not_none(
    pool_size_from_manifest,
    config.get("sampling", {}).get("default_pool_size") if isinstance(config, dict) else None,
)
pool_source = "几何池 manifest" if pool_size_from_manifest is not None else ("配置默认值" if config else "缺失")

initial_size_from_manifest = count_manifest_samples(initial_selection_manifest)
initial_size = first_not_none(
    initial_size_from_manifest,
    config.get("active_learning", {}).get("initial_points") if isinstance(config, dict) else None,
)
initial_source = "初始选点 manifest" if initial_size_from_manifest is not None else ("配置默认值" if config else "缺失")

delta_dataset_size = delta_metadata.get("num_samples") if isinstance(delta_metadata, dict) else None
delta_dataset_source = "delta dataset metadata" if delta_dataset_size is not None else "缺失"

training_success = None
training_source = "缺失"
if isinstance(train_main_status, dict) and isinstance(train_aux_status, dict):
    training_success = bool(train_main_status.get("success")) and bool(train_aux_status.get("success"))
    training_source = "主/辅训练状态文件"
elif isinstance(train_main_status, dict):
    training_success = bool(train_main_status.get("success"))
    training_source = "主模型训练状态文件"

uq_done = None
uq_source = "缺失"
if isinstance(uncertainty_payload, dict):
    uq_done = uncertainty_payload.get("num_samples") is not None
    uq_source = "uncertainty_latest.json"

round1_selected = None
converged = None
selection_source = "缺失"
if isinstance(selection_summary, dict):
    round1_selected = len(selection_summary.get("selected_sample_ids", []))
    if round1_selected == 0 and isinstance(selection_summary.get("selected_samples"), list):
        round1_selected = len(selection_summary.get("selected_samples", []))
    converged = selection_summary.get("converged")
    selection_source = "round_001_selection_summary.json"

overview_df = pd.DataFrame(
    [
        {"指标": "几何池大小", "数值": pool_size, "来源": pool_source},
        {"指标": "初始训练样本数", "数值": initial_size, "来源": initial_source},
        {"指标": "delta 数据集样本数", "数值": delta_dataset_size, "来源": delta_dataset_source},
        {"指标": "是否训练成功", "数值": bool_to_text(training_success), "来源": training_source},
        {"指标": "是否完成 UQ", "数值": bool_to_text(uq_done), "来源": uq_source},
        {"指标": "第 1 轮新增样本数", "数值": round1_selected, "来源": selection_source},
        {"指标": "是否收敛", "数值": bool_to_text(converged), "来源": selection_source},
    ]
)

overview_df


## 3. delta 数据集基本统计

这一格聚焦 `delta_dataset.npz` 与 metadata。我们会看样本数、能量分布、`delta_E` 分布，以及 `delta_F` 的大小分布，并给出基础描述统计表。这样可以先确认数据本身有没有明显异常，再谈模型训练效果。


In [ ]:
# 如果数据集文件缺失，就给出提示并返回一个说明表。
if delta_dataset is None or delta_metadata is None:
    print("提示：缺少 delta_dataset.npz 或 delta_dataset_metadata.json，无法做数据集统计。")
    delta_stats_df = pd.DataFrame({"提示": ["等待数据集文件生成后，再运行本节。"]})
else:
    # 读取能量与力相关数组，并转成后续分析更方便的 numpy 数组。
    e_baseline = np.asarray(delta_dataset.get("E_baseline", []), dtype=float)
    e_target = np.asarray(delta_dataset.get("E_target", []), dtype=float)
    delta_e = np.asarray(delta_dataset.get("delta_E", []), dtype=float)
    delta_f = np.asarray(delta_dataset.get("delta_F", []), dtype=float)

    # 把 delta_F 转成力大小，便于做一维分布统计。
    if delta_f.ndim == 3:
        delta_f_magnitude = np.linalg.norm(delta_f, axis=2).reshape(-1)
    elif delta_f.ndim == 2:
        delta_f_magnitude = np.linalg.norm(delta_f, axis=1).reshape(-1)
    else:
        delta_f_magnitude = np.abs(delta_f).reshape(-1)

    # 组织成表格，再统一用 describe 做基础统计。
    delta_frame = pd.DataFrame(
        {
            "E_baseline": e_baseline,
            "E_target": e_target,
            "delta_E": delta_e,
        }
    )
    delta_stats_df = delta_frame.describe().T
    delta_stats_df.loc["|delta_F|"] = pd.Series(delta_f_magnitude).describe()

    print(f"delta 数据集样本数: {len(delta_frame)}")
    print(f"metadata 记录样本数: {delta_metadata.get('num_samples')}")

    # 画三张最常用的图：总能量、delta_E、delta_F 大小分布。
    if plt is not None:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

        axes[0].hist(e_baseline, bins=20, alpha=0.7, label="E_baseline")
        axes[0].hist(e_target, bins=20, alpha=0.7, label="E_target")
        axes[0].set_title("能量分布")
        axes[0].set_xlabel("Energy")
        axes[0].set_ylabel("Count")
        axes[0].legend()

        axes[1].hist(delta_e, bins=20, color="#2a9d8f", alpha=0.85)
        axes[1].set_title("delta_E 分布")
        axes[1].set_xlabel("delta_E")
        axes[1].set_ylabel("Count")

        axes[2].hist(delta_f_magnitude, bins=20, color="#e76f51", alpha=0.85)
        axes[2].set_title("|delta_F| 分布")
        axes[2].set_xlabel("|delta_F|")
        axes[2].set_ylabel("Count")

        plt.tight_layout()
        plt.show()
    else:
        print("提示：当前环境缺少 matplotlib，本节跳过分布图，只保留统计表。")

delta_stats_df.round(6)


## 4. 训练结果分析

这一格读取 `training_summary.json`，把主模型和辅助模型的训练集、验证集指标整理成表格和柱状图。重点不是把数字背下来，而是学会看哪些数字更关键。


In [ ]:
# 把训练摘要整理成一张更适合比较的表。
def collect_training_metrics(summary: dict | None) -> pd.DataFrame:
    if not isinstance(summary, dict):
        return pd.DataFrame()

    rows = []
    for model_key, model_label in [("main_model", "主模型"), ("aux_model", "辅助模型")]:
        model_metrics = summary.get(model_key)
        if not isinstance(model_metrics, dict):
            continue

        rows.append(
            {
                "模型": model_label,
                "训练能量RMSE": model_metrics.get("subtrain_energy_rmse"),
                "验证能量RMSE": model_metrics.get("validation_energy_rmse"),
                "训练梯度RMSE": model_metrics.get("subtrain_gradient_rmse"),
                "验证梯度RMSE": model_metrics.get("validation_gradient_rmse"),
                "训练PCC": model_metrics.get("subtrain_energy_pcc"),
                "验证PCC": model_metrics.get("validation_energy_pcc"),
            }
        )
    return pd.DataFrame(rows)

training_metrics_df = collect_training_metrics(training_summary)

if training_metrics_df.empty:
    print("提示：缺少 training_summary.json，无法做训练指标分析。")
    training_metrics_df = pd.DataFrame({"提示": ["等待训练完成后，再运行本节。"]})
else:
    # 输出最关键的样本划分信息，帮助解释 train / validation 指标。
    print(f"subtrain 样本数: {training_summary.get('num_subtrain')}")
    print(f"validation 样本数: {training_summary.get('num_validation')}")
    print("怎么看这些指标：")
    print("1. energy RMSE 越小越好，表示能量差预测更准。")
    print("2. gradient RMSE 越小越好，表示力相关预测更准。")
    print("3. PCC 越接近 1 越好，表示预测趋势和真实趋势越一致。")
    print("4. 验证集比训练集更重要，因为它更能反映泛化能力。")
    print("5. 辅助模型默认只学 delta_E，所以梯度列可能为空，这是正常现象。")

    plot_df = training_metrics_df.set_index("模型")
    if plt is not None:
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

        energy_cols = ["训练能量RMSE", "验证能量RMSE"]
        plot_df[energy_cols].plot(kind="bar", ax=axes[0], rot=0, title="能量 RMSE")
        axes[0].set_ylabel("RMSE")

        gradient_cols = ["训练梯度RMSE", "验证梯度RMSE"]
        gradient_plot_df = plot_df[gradient_cols].dropna(how="all")
        if gradient_plot_df.empty:
            axes[1].text(0.5, 0.5, "当前没有可用的梯度指标", ha="center", va="center")
            axes[1].set_axis_off()
        else:
            gradient_plot_df.plot(kind="bar", ax=axes[1], rot=0, title="梯度 RMSE")
            axes[1].set_ylabel("RMSE")

        pcc_cols = ["训练PCC", "验证PCC"]
        plot_df[pcc_cols].plot(kind="bar", ax=axes[2], rot=0, title="能量 PCC")
        axes[2].set_ylim(0, 1.05)
        axes[2].set_ylabel("PCC")

        plt.tight_layout()
        plt.show()
    else:
        print("提示：当前环境缺少 matplotlib，本节跳过柱状图，只保留指标表。")

training_metrics_df.round(6)


## 5. 不确定性分析

这一格读取 `uncertainty_latest.json`，查看 UQ 分布、阈值位置、以及最值得关注的高不确定性样本。主动学习里，这一节往往最直接决定“下一轮该补谁”。


In [ ]:
if not isinstance(uncertainty_payload, dict) or not isinstance(uncertainty_payload.get("samples"), list):
    print("提示：缺少 uncertainty_latest.json，无法做不确定性分析。")
    uncertainty_topn_df = pd.DataFrame({"提示": ["等待 UQ 完成后，再运行本节。"]})
else:
    # 把逐样本 UQ 结果整理成表，并按不确定性从高到低排序。
    uncertainty_df = pd.DataFrame(uncertainty_payload.get("samples", []))
    uncertainty_df = uncertainty_df.sort_values("uncertainty", ascending=False).reset_index(drop=True)
    uq_threshold = uncertainty_payload.get("uq_threshold")

    if uncertainty_df.empty:
        print("提示：不确定性文件存在，但 samples 为空。")
        uncertainty_topn_df = pd.DataFrame({"提示": ["当前没有可供分析的 UQ 样本。"]})
    else:
        # 统计高于阈值的样本数量和比例。
        if uq_threshold is None:
            high_uq_mask = np.ones(len(uncertainty_df), dtype=bool)
        else:
            high_uq_mask = uncertainty_df["uncertainty"].to_numpy(dtype=float) > float(uq_threshold)

        num_high_uq = int(high_uq_mask.sum())
        ratio_high_uq = num_high_uq / len(uncertainty_df)
        uncertainty_topn_df = uncertainty_df.head(min(TOP_N, len(uncertainty_df))).copy()

        print(f"UQ 样本总数: {len(uncertainty_df)}")
        print(f"UQ 阈值: {uq_threshold}")
        print(f"高于阈值样本数: {num_high_uq}")
        print(f"高于阈值样本比例: {ratio_high_uq:.2%}")

        # 左图看整体分布，右图看排序后的阈值位置。
        if plt is not None:
            fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

            axes[0].hist(uncertainty_df["uncertainty"], bins=20, color="#457b9d", alpha=0.85)
            if uq_threshold is not None:
                axes[0].axvline(float(uq_threshold), color="#d62828", linestyle="--", label="threshold")
                axes[0].legend()
            axes[0].set_title("UQ 分布直方图")
            axes[0].set_xlabel("uncertainty")
            axes[0].set_ylabel("Count")

            axes[1].plot(
                np.arange(1, len(uncertainty_df) + 1),
                uncertainty_df["uncertainty"].to_numpy(dtype=float),
                marker="o",
                markersize=3,
                linewidth=1,
            )
            if uq_threshold is not None:
                axes[1].axhline(float(uq_threshold), color="#d62828", linestyle="--", label="threshold")
                axes[1].legend()
            axes[1].set_title("按不确定性排序后的阈值可视化")
            axes[1].set_xlabel("样本排序")
            axes[1].set_ylabel("uncertainty")

            plt.tight_layout()
            plt.show()
        else:
            print("提示：当前环境缺少 matplotlib，本节跳过 UQ 图，只保留统计表。")

        uncertainty_topn_df = uncertainty_topn_df[
            [
                column
                for column in [
                    "sample_id",
                    "uncertainty",
                    "predicted_delta_E_main",
                    "predicted_delta_E_aux",
                    "exceeds_threshold",
                ]
                if column in uncertainty_topn_df.columns
            ]
        ]

uncertainty_topn_df.round(6)


## 6. 第 1 轮选点分析

这一格读取 `round_001_selection_summary.json`，看第一轮到底挑了哪些样本、挑了多少个、当前高不确定性比例是多少，以及流程是否已经满足收敛条件。


In [ ]:
if not isinstance(selection_summary, dict):
    print("提示：缺少 round_001_selection_summary.json，无法分析第 1 轮选点。")
    selection_df = pd.DataFrame({"提示": ["等待选点摘要文件生成后，再运行本节。"]})
else:
    # 优先使用完整的 selected_samples；如果没有，再退回 sample_id 列表。
    if isinstance(selection_summary.get("selected_samples"), list) and selection_summary.get("selected_samples"):
        selection_df = pd.DataFrame(selection_summary.get("selected_samples", []))
    else:
        selection_df = pd.DataFrame({"sample_id": selection_summary.get("selected_sample_ids", [])})

    num_selected = len(selection_df)
    uncertain_ratio = selection_summary.get("uncertain_ratio")
    converged_flag = selection_summary.get("converged")

    print(f"选点数量: {num_selected}")
    if uncertain_ratio is not None:
        print(f"高不确定性比例: {float(uncertain_ratio):.2%}")
    else:
        print("高不确定性比例: 未提供")
    print(f"是否收敛: {bool_to_text(converged_flag)}")

    # 保留最常用的展示列，方便直接放进汇报。
    preferred_columns = ["sample_id", "uncertainty", "predicted_delta_E_main", "predicted_delta_E_aux"]
    available_columns = [column for column in preferred_columns if column in selection_df.columns]
    if available_columns:
        selection_df = selection_df[available_columns]

selection_df.round(6)


## 7. 面向汇报的结论区

这一格会根据前面的结果自动拼出几条中文结论，方便你直接放进组会汇报、实验记录或周报里。它不是替代人工判断，而是帮你先把“能直接说的话”整理出来。


In [ ]:
conclusions = []

# 根据训练和 UQ 状态判断是否形成了完整闭环。
if training_success is True and uq_done is True:
    conclusions.append("本轮已完成第一轮闭环。")
elif training_success is True and uq_done is not True:
    conclusions.append("训练已经完成，但还需要补做不确定性评估，闭环才算完整。")
else:
    conclusions.append("当前工作区尚未提供完整训练结果，是否完成闭环仍需结合结果文件判断。")

# 如果已经有选点摘要，就优先给出收敛判断。
if isinstance(selection_summary, dict) and selection_summary.get("uncertain_ratio") is not None:
    current_ratio = float(selection_summary.get("uncertain_ratio"))
    if current_ratio < 0.05:
        conclusions.append(f"当前高不确定性比例为 {current_ratio:.2%}，低于 5%，满足收敛条件。")
    else:
        conclusions.append(f"当前高不确定性比例为 {current_ratio:.2%}，仍高于 5%，建议继续下一轮主动学习。")

# 如果已经选出新样本，就把下一步建议直接写出来。
if isinstance(selection_summary, dict):
    selected_ids = selection_summary.get("selected_sample_ids", [])
    if selected_ids:
        sample_text = "、".join(map(str, selected_ids[:5]))
        conclusions.append(f"若继续第二轮，可优先标注这 {len(selected_ids)} 个样本：{sample_text}。")

# 用数据集规模补一句背景说明，方便汇报时交代当前基线规模。
if delta_dataset_size is not None:
    conclusions.append(f"当前 delta 数据集规模为 {delta_dataset_size} 个样本，可作为后续轮次比较的基线。")

if not conclusions:
    conclusions.append("当前缺少足够的结果文件，建议先完成标注、训练和 UQ，再回来运行本 notebook。")

conclusions_df = pd.DataFrame({"自动结论": conclusions})

for idx, item in enumerate(conclusions, start=1):
    print(f"{idx}. {item}")

conclusions_df


## 8. 复用说明

这份模板默认绑定当前仓库的推荐路径，但它本质上是可复用的。以后如果你继续跑第二轮，或者把流程迁移到别的体系，优先改下面这些位置即可。


In [ ]:
# 把最常改的地方整理成一张表，后续复用时按这张表改最省事。
reuse_df = pd.DataFrame(
    [
        {
            "复用场景": "继续分析第二轮",
            "优先修改": "把 round_001_selection_summary.json 改成 round_002_selection_summary.json，或新增一个轮次变量统一控制。",
            "当前设置": as_relative(PATHS["round_selection"]),
        },
        {
            "复用场景": "更换体系但仍沿用本仓库结构",
            "优先修改": "确认 REPO_ROOT 自动定位是否正确；重点检查 config、delta dataset、training summary、uncertainty 文件路径。",
            "当前设置": str(REPO_ROOT),
        },
        {
            "复用场景": "只分析结果文件，不跑全流程",
            "优先修改": "至少准备 training_summary.json、uncertainty_latest.json、round_xxx_selection_summary.json；缺少 delta_dataset.npz 时会跳过数据集统计。",
            "当前设置": "见本 notebook 的 PATHS 与 OPTIONAL_PATHS",
        },
    ]
)

print("如果以后换轮次，最先改的是选点摘要文件名；如果换体系，最先检查的是路径。")
reuse_df
